# 🌳 Guía General: Base de Datos FIA Chirimoyo
## Diseño, ETL y Carga en SQL Server

---

## 📋 Contenido

1. [Arquitectura de la Base de Datos](#arquitectura)
2. [Flujo de Trabajo ETL](#flujo-etl)
3. [Ejecución Paso a Paso](#pasos)
4. [Consultas Analíticas](#consultas)
5. [Troubleshooting](#troubleshooting)

---

## 🏗️ Arquitectura de la Base de Datos {#arquitectura}

### Concepto: Los 3 Esquemas

Una base de datos **profesional** organiza datos en capas según su madurez:

```
CSV (datos crudos)
    ↓
[STG - STAGING]  ← Datos exactamente como llegan del sensor
    ↓ (validación + transformación)
[DIM - DIMENSIONS] ← Catálogos (tratamientos, ubicaciones, etc)
    ↓
[FACT - FACTS] ← Datos listos para análisis agrícola
```

### Tablas en Nuestro Modelo

#### 1️⃣ **dim.tratamiento** (Dimensión)
```
id_tratamiento  | nombre_tratamiento | descripción
────────────────┼────────────────────┼──────────────────────────
1               | Campo              | Cultivo al aire libre
2               | Invernadero        | Cultivo bajo invernadero
```

**¿Por qué es una DIMENSIÓN?**
- Datos de referencia que no cambian frecuentemente
- Se consulta junto con mediciones para contexto
- Facilita cambios de nombres sin afectar datos históricos

---

#### 2️⃣ **stg.mediciones_raw** (Staging)
```
id_raw | fecha      | tratamiento | temp_min | temp_media | temp_max | ... | fecha_carga
───────┼────────────┼─────────────┼──────────┼────────────┼──────────┼─────┼─────────────
1      | 2023-11-11 | Campo       | 10.5     | 16.1       | 21.7     | ... | 2026-06-10
2      | 2023-11-12 | Campo       | 7.9      | 14.8       | 21.7     | ... | 2026-06-10
```

**Características:**
- Almacena datos **exactamente** como vienen del CSV
- Permite auditoría: sabemos qué llegó del sensor
- Se limpia después de transformación (truncate)
- Tipos de datos flexibles (DECIMAL para números)

**CHECK Constraints** (validación a nivel DB):
```sql
CHECK (temp_min >= -10 AND temp_min <= 50)
CHECK (hr_min >= 0 AND hr_min <= 100)
CHECK (precipitaciones_mm >= 0)
```

#### 3️⃣ **fact.mediciones_climaticas** (Fact Table Principal)
```
id_medicion | id_tratamiento | fecha      | temp_min | temp_media | ... | es_valido | notas_qc
────────────┼────────────────┼────────────┼──────────┼────────────┼─────┼───────────┼─────────
1           | 1              | 2023-11-11 | 10.5     | 16.1       | ... | 1         | NULL
2           | 1              | 2023-11-12 | 7.9      | 14.8       | ... | 1         | NULL
```

**Diferencias vs staging:**
- ✅ Referencia FK a `dim.tratamiento` (integridad)
- ✅ CHECK constraints más restrictivos
- ✅ Campo `es_valido` para control de calidad
- ✅ Índices para consultas rápidas

**Validación en ETL:**
```python
# En Python, antes de cargar:
if temp_min <= temp_media <= temp_max:
    es_valido = True
else:
    es_valido = False
    notas_qc = "Jerarquía de temperatura invertida"
```

---

#### 4️⃣ **fact.dpv** y **fact.horas_frio** (Fact Tables Especializadas)

Tablas enfocadas en variables agrícolas críticas:

**fact.dpv** (Déficit de Presión de Vapor)
- Para estudios de estrés hídrico
- DPV > 3 kPa = estrés hídrico en muchos cultivos

**fact.horas_frio** (Acumulación de Horas Frío)
- Para chirimoya: requiere ~400-600 horas < 7.2°C
- Facilita monitoreo en tiempo real
- Columna calculada `temporada` (2023-2024)

---

## 🔄 Flujo de Trabajo ETL {#flujo-etl}

### ¿Qué es ETL?

**E**xtract → **T**ransform → **L**oad

```
SENSOR EN CAMPO (temperatura, humedad, etc.)
        ↓
    EXCEL/CSV (archivo con datos crudos)
        ↓
    [PYTHON] - EXTRACCIÓN
        - Leer .xlsx
        - Limpiar nombres de columnas
        - Validar tipos de datos
        ↓
    [PYTHON] - TRANSFORMACIÓN
        - Normalizar valores
        - Generar CSVs con formato chileno
        - Reportar anomalías
        ↓
    CSV CON FORMATO CHILENO
    (;  UTF-8 BOM, CRLF)
        ↓
    [SQL SERVER] - CARGA
        - BULK INSERT a stg.mediciones_raw
        - Validación adicional
        - Transformación a fact.*
        ↓
    BASE DE DATOS LISTA PARA ANÁLISIS

```

---

## 📊 Convenciones Chilenas para CSV {#convenciones}

El formato CSV generado sigue estándares chilenos:

| Aspecto | Valor | Razón |
|---------|-------|-------|
| **Delimitador** | `;` (punto y coma) | Estándar EN usado en Excel Chile |
| **Codificación** | UTF-8 con BOM | Compatible con acentos: "temperatura" |
| **Decimal** | `,` (coma) | 15,5°C no 15.5°C |
| **Líneas** | CRLF (\r\n) | Compatibilidad Windows |

**Ejemplo de CSV válido:**
```
id_raw;fecha;tratamiento;temp_min;temp_media;temp_max;hr_min;hr_media;hr_max
1;2023-11-11;Campo;10,5;16,1;21,7;58,3;78,8;99,0
2;2023-11-12;Campo;7,9;14,8;21,7;52,5;73,4;94,1
```

---

---

## 🚀 Ejecución Paso a Paso {#pasos}

### Requisitos
- SQL Server Express (local)
- Python 3.13 con pandas
- SSMS (SQL Server Management Studio)
- Archivos .xlsx con datos

---

### Paso 1: Ejecutar Pipeline Python

```bash
cd /ruta/al/proyecto
python3 etl_chirimoyo_v3.py
```

**Salida esperada:**
```
✓ Datos cargados: 1860 filas × 13 columnas
✓ Archivo completo: mediciones_completo.csv
✓ Tratamiento CAMPO: mediciones_campo.csv
✓ Tratamiento INVERNADERO: mediciones_invernadero.csv
✓ Tabla de referencia TRATAMIENTOS: dim_tratamiento.csv
```

**Archivos generados en `/mnt/user-data/outputs/`:**
1. `mediciones_completo.csv` (1860 registros)
2. `mediciones_campo.csv` (930 registros)
3. `mediciones_invernadero.csv` (930 registros)
4. `dim_tratamiento.csv` (2 registros - catálogo)

---

### Paso 2: Crear Estructura en SQL Server

En **SSMS**, ejecutar scripts en orden:

```sql
-- Opción A: Ejecutar scripts individuales
1. Abrir "01_crear_esquemas.sql" → F5
2. Abrir "02_crear_tablas.sql" → F5
3. Abrir "03_procedimientos_etl.sql" → F5
```

O bien:

```sql
-- Opción B: Crear base de datos primero (si no existe)
CREATE DATABASE FIA_Chirimoyo
GO
USE FIA_Chirimoyo
GO

-- Luego ejecutar los scripts
```

---
**Verificación:** Ver estructura creada
```sql
-- Esquemas
SELECT name FROM sys.schemas WHERE name IN ('stg', 'dim', 'fact')

-- Tablas
SELECT SCHEMA_NAME(schema_id), name FROM sys.tables 
WHERE SCHEMA_NAME(schema_id) IN ('stg', 'dim', 'fact')
ORDER BY SCHEMA_NAME(schema_id), name
```

---

---
### Paso 3: Cargar Datos

En **SSMS**, ejecutar:

```sql
USE FIA_Chirimoyo
GO

-- 3.1 Cargar catálogos
EXEC sp_cargar_tratamientos @debug = 1

-- 3.2 Cargar mediciones desde CSV
-- IMPORTANTE: Ajustar ruta según tu sistema operativo
EXEC sp_cargar_mediciones_raw 
    @ruta_csv = 'C:\ruta\a\mediciones_completo.csv',
    @debug = 1

-- 3.3 Transformar a tablas fact
EXEC sp_transformar_a_fact @debug = 1
```

**Rutas según SO:**
- **Windows:** `C:\Users\usuario\mediciones_campo.csv`
- **Linux:** `/home/usuario/mediciones_campo.csv`
- **Mac:** `/Users/usuario/mediciones_campo.csv`

---

---

### Paso 4: Validar Carga

```sql
-- Contar registros en cada tabla
SELECT 'stg.mediciones_raw' AS Tabla, COUNT(*) AS Registros 
FROM stg.mediciones_raw
UNION ALL
SELECT 'fact.mediciones_climaticas', COUNT(*) FROM fact.mediciones_climaticas
UNION ALL
SELECT 'fact.dpv', COUNT(*) FROM fact.dpv
UNION ALL
SELECT 'fact.horas_frio', COUNT(*) FROM fact.horas_frio

-- Verificar que los tratamientos fueron referenciados correctamente
SELECT t.nombre_tratamiento, COUNT(*) AS Mediciones
FROM fact.mediciones_climaticas mc
INNER JOIN dim.tratamiento t ON mc.id_tratamiento = t.id_tratamiento
GROUP BY t.nombre_tratamiento
```

---

---

## 📈 Consultas Analíticas {#consultas}

### Consulta 1: Comparación Clima Campo vs Invernadero

```sql
SELECT 
    t.nombre_tratamiento,
    COUNT(*) AS Total_Registros,
    CAST(AVG(mc.temp_media) AS DECIMAL(5,2)) AS Temp_Media_Promedio,
    CAST(AVG(mc.hr_media) AS DECIMAL(5,2)) AS HR_Media_Promedio,
    CAST(MIN(mc.fecha) AS DATE) AS Fecha_Inicio,
    CAST(MAX(mc.fecha) AS DATE) AS Fecha_Fin
FROM fact.mediciones_climaticas mc
INNER JOIN dim.tratamiento t ON mc.id_tratamiento = t.id_tratamiento
WHERE mc.es_valido = 1  -- Solo datos válidos
GROUP BY t.nombre_tratamiento, t.id_tratamiento
ORDER BY t.id_tratamiento
```

---

---
### Consulta 2: Seguimiento de Horas Frío (Crítico para Chirimoya)

```sql
-- ¿Cuántas horas frío acumuló cada tratamiento?
SELECT 
    t.nombre_tratamiento,
    fh.temporada,
    CAST(MAX(fh.hf_acumuladas) AS DECIMAL(8,1)) AS Horas_Frio_Acumuladas,
    CASE 
        WHEN MAX(fh.hf_acumuladas) >= 500 THEN '✓ Suficientes para florecer'
        WHEN MAX(fh.hf_acumuladas) >= 400 THEN '⚠ Cercano a mínimo'
        ELSE '✗ Insuficientes'
    END AS Estado_Chirimoya
FROM fact.horas_frio fh
INNER JOIN dim.tratamiento t ON fh.id_tratamiento = t.id_tratamiento
GROUP BY t.nombre_tratamiento, fh.temporada, t.id_tratamiento
ORDER BY fh.temporada DESC, t.id_tratamiento
```
---

---
### Consulta 3: Registros con Posibles Errores

```sql
-- Mostrar mediciones que no pasaron validación
SELECT 
    t.nombre_tratamiento,
    mc.fecha,
    mc.temp_min,
    mc.temp_media,
    mc.temp_max,
    mc.notas_qc AS Motivo_Rechazo
FROM fact.mediciones_climaticas mc
INNER JOIN dim.tratamiento t ON mc.id_tratamiento = t.id_tratamiento
WHERE mc.es_valido = 0  -- Registros con problemas
ORDER BY mc.fecha DESC
```

---

---

### Consulta 4: Exportar Datos para Análisis en Python/R

```sql
-- Exportar mediciones de campo a análisis estadístico
SELECT 
    mc.fecha,
    mc.temp_min,
    mc.temp_media,
    mc.temp_max,
    mc.hr_min,
    mc.hr_media,
    mc.hr_max,
    mc.precipitaciones_mm,
    fd.dpv_kpa,
    fh.hf_acumuladas
FROM fact.mediciones_climaticas mc
LEFT JOIN fact.dpv fd 
    ON mc.id_tratamiento = fd.id_tratamiento 
    AND mc.fecha = fd.fecha
LEFT JOIN fact.horas_frio fh 
    ON mc.id_tratamiento = fh.id_tratamiento 
    AND mc.fecha = fh.fecha
WHERE mc.id_tratamiento = 1  -- Campo
ORDER BY mc.fecha
```

---

---

## 🔧 Troubleshooting {#troubleshooting}

### ❌ Error: "El archivo no se encuentra"
```
Msg 4860 - Cannot bulk load. The file C:\ruta\archivo.csv does not exist
```

**Solución:**
1. Verificar que el archivo existe: `dir C:\ruta\archivo.csv`
2. Verificar permisos: el usuario SQL Server debe poder leer el archivo
3. Usar rutas completas (no relativas)
4. En Linux: usar `/` no `\`

---


---

### ❌ Error: "Syntax error in CSV"
```
Msg 4863 - Bulk load data conversion error
```

**Posibles causas:**
- ❌ Delimitador incorrecto (usar `;` no `,`)
- ❌ Encoding incorrecto (debe ser UTF-8 con BOM)
- ❌ Valores numéricos con formato incorrecto (`,` para decimales)

**Verificar CSV:**
```bash
# En Windows PowerShell:
Get-Content archivo.csv -Encoding utf8 | Select-Object -First 5

# En Linux/Mac:
file archivo.csv  # Debe mostrar "UTF-8"
head -5 archivo.csv
```

---

---

### ❌ Error: "FK constraint violation"
```
Msg 547 - INSERT statement conflicted with FOREIGN KEY constraint
```

**Causa:** El tratamiento en CSV no coincide con dim.tratamiento

**Solución:**
```sql
-- Verificar tratamientos disponibles
SELECT id_tratamiento, nombre_tratamiento FROM dim.tratamiento

-- Verificar tratamientos en CSV
SELECT DISTINCT tratamiento FROM stg.mediciones_raw

-- Ajustar nombres si es necesario:
UPDATE stg.mediciones_raw 
SET tratamiento = 'Campo' 
WHERE LOWER(TRIM(tratamiento)) = 'campo'
```

---

---

### ❌ Error: "Numeric value out of range"
```
Msg 206 - Operand type clash: The binary operator check constraints
```

**Causa:** Dato viola CHECK constraint de la tabla

**Solución:**
```sql
-- Buscar registro problemático:
SELECT fecha, temp_min, temp_media, temp_max 
FROM stg.mediciones_raw
WHERE temp_max > 50  -- Fuera de rango

-- Opción 1: Corregir en CSV y recargar
-- Opción 2: Marcar como inválido en ETL
```

---

---

## 📚 Conceptos Clave para Estudiantes

### ¿Por qué separar en 3 esquemas?

| Esquema | Propósito | Ejemplo de Uso |
|---------|-----------|-----------------|
| **STG** | Auditoría y trazabilidad | "¿Qué datos llegaron del CSV?" |
| **DIM** | Contexto y referencia | "¿Cuál es el nombre del tratamiento?" |
| **FACT** | Análisis y reportes | "¿Cuál fue la temperatura promedio?" |

### ¿Cuándo usar BULK INSERT vs INSERT?

| Situación | Método | Razón |
|-----------|--------|-------|
| Cargar **miles** de registros | BULK INSERT | Más rápido, optimizado para SQL Server |
| Cargar **pocos** registros | INSERT con VALUES | Simple, directo, mejor para cambios puntuales |
| Carga **regular** desde CSV | BULK INSERT | Automatizable con procedures |

---
### ¿Por qué "stg" se limpia después?

```sql
-- Flujo correcto:
BULK INSERT stg.mediciones_raw FROM 'archivo.csv'
-- Ahora: 1860 registros en stg

EXEC sp_transformar_a_fact
-- Ahora: 1860 registros en fact.*, stg sin cambios

TRUNCATE TABLE stg.mediciones_raw
-- Ahora: stg vacío, lista para próxima carga
```

**Ventaja:** Cada carga es independiente, sin acumulación de datos.

---

---

## 🎓 Para Enseñanza en Aula

### Ejercicio 1: Identificar anomalías
```sql
-- Buscar registros donde la jerarquía de temperatura está invertida
SELECT fecha, tratamiento, temp_min, temp_media, temp_max
FROM stg.mediciones_raw
WHERE temp_min > temp_media OR temp_media > temp_max
```

### Ejercicio 2: Calcular DPV faltante
Si DPV no está disponible, puede estimarse desde T y HR:
```sql
-- Pseudocódigo: DPV = e_s(T) - RH% * e_s(T) / 100
-- Implementar en Python o SQL
```

### Ejercicio 3: Comparación visual
Exportar datos a Python y graficar:
```python
import pandas as pd
import matplotlib.pyplot as plt

# Leer desde SQL o CSV
df_campo = pd.read_csv('mediciones_campo.csv', sep=';', decimal=',')
df_inver = pd.read_csv('mediciones_invernadero.csv', sep=';', decimal=',')

# Gráfico de temperaturas
plt.plot(df_campo['fecha'], df_campo['temp_media'], label='Campo')
plt.plot(df_inver['fecha'], df_inver['temp_media'], label='Invernadero')
plt.legend()
plt.show()
```

---

## ✅ Checklist Final

- [ ] Scripts Python ejecutados sin errores
- [ ] Archivos CSV generados en `/outputs/`
- [ ] Base de datos creada en SQL Server
- [ ] Esquemas (stg, dim, fact) creados
- [ ] Tratamientos cargados en dim.tratamiento
- [ ] Datos cargados en stg.mediciones_raw
- [ ] Datos transformados a fact tables
- [ ] Consultas analíticas funcionan correctamente
- [ ] Horas frío acumuladas comparadas (campo > invernadero)

---

## 📞 Preguntas Frecuentes

**P: ¿Por qué no cargar directamente al fact sin usar staging?**
R: El staging permite auditoría. Si algo falla, sabemos exactamente qué datos llegaron del sensor.

**P: ¿Qué hacer si hay registros duplicados?**
R: El SP usa NOT EXISTS para evitar duplicados. Si existen, verificar fecha + tratamiento.

**P: ¿Cómo actualizar datos históricos?**
R: No se deben actualizar fact tables. En su lugar, truncate + recargar desde CSV limpio.

**P: ¿Puedo usar esto en producción?**
R: Sí, pero agregar más validaciones y logging. Este es un modelo educativo.

---

**Última actualización:** 10-06-2026  